In [1]:
!pip install pandas numpy nltk scikit-learn

In [2]:
import pandas as pd
import nltk
import json
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [3]:
nltk.download('punkt_tab')   
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\GOWTHAMSANJAY\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\GOWTHAMSANJAY\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
import json
data = []
with open('News_Category_Dataset_v3.json', encoding='utf-8') as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except:
            continue   
df = pd.DataFrame(data)
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [7]:
df = df[['headline', 'category']]
df.head()

,headline,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [8]:
categories = ['SPORTS', 'POLITICS', 'TECH', 'BUSINESS']

df = df[df['category'].isin(categories)]
df['category'].value_counts()

category
POLITICS    13480
SPORTS        816
BUSINESS      452
TECH          170
Name: count, dtype: int64

In [9]:
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    words = nltk.word_tokenize(text.lower())
    words = [stemmer.stem(w) for w in words if w.isalpha() and w not in stop_words]
    return " ".join(words)

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


In [10]:
df['clean_text'] = df['headline'].apply(preprocess)
df.head()

,headline,category,clean_text
13,Twitch Bans Gambling Sites After Streamer Scam...,TECH,twitch ban gambl site streamer scam folk
17,"Maury Wills, Base-Stealing Shortstop For Dodge...",SPORTS,mauri will shortstop dodger die
21,Biden Says U.S. Forces Would Defend Taiwan If ...,POLITICS,biden say forc would defend taiwan china invad
24,‘Beautiful And Sad At The Same Time’: Ukrainia...,POLITICS,beauti sad time ukrainian cultur festiv take d...
26,"Las Vegas Aces Win First WNBA Title, Chelsea G...",SPORTS,la vega ace win first wnba titl chelsea gray n...


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
min_count = df['category'].value_counts().min()

df_balanced = df.groupby('category').apply(lambda x: x.sample(min_count)).reset_index(drop=True)

df_balanced['category'].value_counts()
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df_balanced['clean_text'])
y = df_balanced['category']

C:\Users\GOWTHAMSANJAY\AppData\Local\Temp\ipykernel_6264\2615971993.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('category').apply(lambda x: x.sample(min_count)).reset_index(drop=True)


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [24]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6323529411764706


In [26]:
def predict_news(text):
    text = preprocess(text)
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

predict_news("India wins cricket world cup")

'SPORTS'